# SFT (no-image setting) + GRPO

4. Build multi-turn trajectory dataset

各 train case で trajectory を programmatic に組む:
- system + user (image pair + task)
- assistant tool_call(compute_index_delta) + tool obs (全 6 index)
- assistant tool_call(analyze, ...) + tool ack
- assistant tool_call(submit_to_ground / drop, reason)


In [ ]:
import yaml, random, collections, json
from datasets import Dataset

INDICES = ["NBR", "NDVI", "NDWI", "MNDWI", "NDBI", "NDSI"]


def read_compute_index_delta(case_id, index):
    path = f"{PRECOMP_ROOT}/{case_id}/compute_index_delta/{index}.stats.yaml"
    if not os.path.isfile(path): return None
    try:
        return (yaml.safe_load(open(path)).get("response") or {}).get("delta_stats") or {}
    except Exception:
        return None


def case_features(case_id):
    deltas = {}
    for idx in INDICES:
        d = read_compute_index_delta(case_id, idx)
        if d is None: return None
        deltas[idx] = d
    return deltas


def _level(v):
    if v is None: return "NONE"
    if v >= 0.30: return "STRONG"
    if v >= 0.10: return "MODERATE"
    return "NONE"

LEVEL_RANK = {"NONE": 0, "MODERATE": 1, "STRONG": 2}


def tags_of(feats):
    nbr   = feats["NBR"]; ndvi  = feats["NDVI"]; ndwi  = feats["NDWI"]
    mndwi = feats["MNDWI"]; ndbi  = feats["NDBI"]; ndsi  = feats["NDSI"]
    return {
        "burn":    _level(nbr.get("frac_decrease_strong")),
        "veg":     _level(ndvi.get("frac_decrease_strong")),
        "water":   max([_level(ndwi.get("frac_decrease_strong")),  _level(ndwi.get("frac_increase_strong")),
                        _level(mndwi.get("frac_decrease_strong")), _level(mndwi.get("frac_increase_strong"))],
                       key=lambda x: LEVEL_RANK[x]),
        "builtup": _level(ndbi.get("frac_increase_strong")),
        "snow":    max([_level(ndsi.get("frac_decrease_strong")), _level(ndsi.get("frac_increase_strong"))],
                       key=lambda x: LEVEL_RANK[x]),
    }


def build_observation(feats):
    tags = tags_of(feats)
    means = {idx: feats[idx].get("mean") or 0.0 for idx in INDICES}
    return (
        f"burn_or_biomass_loss: {tags['burn']} (NBR mean={means['NBR']:+.2f})\n"
        f"vegetation_loss:      {tags['veg']} (NDVI mean={means['NDVI']:+.2f})\n"
        f"water_change:         {tags['water']} (NDWI mean={means['NDWI']:+.2f}, MNDWI mean={means['MNDWI']:+.2f})\n"
        f"built_up_change:      {tags['builtup']} (NDBI mean={means['NDBI']:+.2f})\n"
        f"snow_change:          {tags['snow']} (NDSI mean={means['NDSI']:+.2f})"
    )


def category_of(case):
    cid = case["id"]
    if cid.startswith("hardneg_volcano"): return "neg_hard_volcano"
    if cid.startswith("hardneg_forest"):  return "neg_hard_forest"
    if cid.startswith("hardneg_preburn"): return "neg_hard_preburn"
    if cid.startswith("neg__"):           return "neg_soft"
    if case.get("label") == "fire":          return "pos_fire"
    if case.get("label") == "volcanic":      return "pos_volcanic"
    if case.get("label") == "deforestation": return "pos_deforestation"
    return "OTHER"


SYS_PROMPT = (
    "You are an onboard satellite operator agent on Earth-observation duty.\n\n"
    "You are shown a Sentinel-2 image pair (before / after) of the same location. "
    "Decide whether to transmit a report to ground (submit_to_ground) or discard the data (drop).\n\n"
    "Use compute_index_delta() (with no arguments) to get all 6 spectral indices at once. "
    "Then call analyze() to commit your reasoning, then call submit_to_ground() or drop()."
)


TOOLS = [
    {"type": "function", "function": {
        "name": "compute_index_delta",
        "description": "Returns spectral delta tags. Call with no arguments to get all 6 indices.",
        "parameters": {
            "type": "object",
            "properties": {"index": {"type": "string",
                "enum": ["all", "NBR", "NDVI", "NDWI", "MNDWI", "NDBI", "NDSI"]}},
            "required": [],
        },
    }},
    {"type": "function", "function": {
        "name": "analyze",
        "description": "Commit reasoning before terminal action.",
        "parameters": {
            "type": "object",
            "properties": {
                "evidence":     {"type": "string"},
                "interpretation": {"type": "string", "enum": ["significant_change", "no_significant_change"]},
                "recommended_action": {"type": "string", "enum": ["submit_to_ground", "drop"]},
            },
            "required": ["evidence", "interpretation", "recommended_action"],
        },
    }},
    {"type": "function", "function": {
        "name": "submit_to_ground",
        "description": "Transmit the report.",
        "parameters": {"type": "object", "properties": {"reason": {"type": "string"}}, "required": ["reason"]},
    }},
    {"type": "function", "function": {
        "name": "drop",
        "description": "Discard the data.",
        "parameters": {"type": "object", "properties": {"reason": {"type": "string"}}, "required": ["reason"]},
    }},
]


def build_trajectory(case, feats):
    cid = case["id"]
    before = f"{DATA_ROOT}/curated_pairs/{cid}/before.png"
    after  = f"{DATA_ROOT}/curated_pairs/{cid}/after.png"
    if not (os.path.isfile(before) and os.path.isfile(after)): return None
    is_pos = (case.get("type") == "positive")
    label = case.get("label", "")
    obs = build_observation(feats)
    tags = tags_of(feats)

    non_none_tags = [f"{k}={v}" for k, v in tags.items() if v != "NONE"]
    evidence = ", ".join(non_none_tags) if non_none_tags else "all spectral signals are NONE"

    if is_pos:
        interpretation = "significant_change"
        action = "submit_to_ground"
        if label == "fire":           reason = f"Fire scar inferred from {evidence}"
        elif label == "volcanic":     reason = f"Volcanic activity inferred from {evidence}"
        elif label == "deforestation": reason = f"Deforestation inferred from {evidence}"
        else:                          reason = f"Significant change: {evidence}"
    else:
        interpretation = "no_significant_change"
        action = "drop"
        reason = f"No actionable change: {evidence}"

    # arguments: json.dumps(dict) → string. prime-rl 0.4.0 calls json.loads() unconditionally.
    msgs = [
        {"role": "system",
         "content": [{"type": "text", "text": SYS_PROMPT}],
         "tool_calls": None, "tool_call_id": None, "name": None},
        {"role": "user",
         "content": [
             {"type": "text",  "text": "Before image (previous pass):"},
             {"type": "image", "path": before},
             {"type": "text",  "text": "After image (current pass, same location):"},
             {"type": "image", "path": after},
             {"type": "text",  "text": "Investigate with your tools, then decide submit_to_ground or drop."},
         ],
         "tool_calls": None, "tool_call_id": None, "name": None},
        {"role": "assistant",
         "content": [{"type": "text", "text": ""}],
         "tool_calls": [{"id": "c1", "type": "function",
            "function": {"name": "compute_index_delta", "arguments": json.dumps({})}}],
         "tool_call_id": None, "name": None},
        {"role": "tool",
         "content": [{"type": "text", "text": obs}],
         "tool_calls": None, "tool_call_id": "c1", "name": "compute_index_delta"},
        {"role": "assistant",
         "content": [{"type": "text", "text": ""}],
         "tool_calls": [{"id": "c2", "type": "function",
            "function": {"name": "analyze",
                         "arguments": json.dumps({"evidence": evidence,
                                                  "interpretation": interpretation,
                                                  "recommended_action": action})}}],
         "tool_call_id": None, "name": None},
        {"role": "tool",
         "content": [{"type": "text", "text": f"noted, call {action}"}],
         "tool_calls": None, "tool_call_id": "c2", "name": "analyze"},
        {"role": "assistant",
         "content": [{"type": "text", "text": ""}],
         "tool_calls": [{"id": "c3", "type": "function",
            "function": {"name": action, "arguments": json.dumps({"reason": reason})}}],
         "tool_call_id": None, "name": None},
    ]
    return {"messages": msgs, "tools": json.dumps(TOOLS), "case_id": cid, "category": category_of(case)}


canonical = yaml.safe_load(open(f"{DATA_ROOT}/canonical_dataset.yaml"))
cases = canonical["cases"]
print(f"total canonical cases: {len(cases)}")

all_examples = []
for c in cases:
    feats = case_features(c["id"])
    if feats is None: continue
    ex = build_trajectory(c, feats)
    if ex is not None: all_examples.append(ex)
print(f"usable examples: {len(all_examples)}")

by_cat = collections.defaultdict(list)
for ex in all_examples:
    by_cat[ex["category"]].append(ex)
random.seed(0)
train_examples, test_examples = [], []
for cat, lst in by_cat.items():
    random.shuffle(lst)
    cut = int(len(lst) * 0.8)
    train_examples.extend(lst[:cut])
    test_examples.extend(lst[cut:])
print(f"train: {len(train_examples)}, test: {len(test_examples)}")

print()
print("=== sample trajectory (first train) ===")
ex = train_examples[0]
print(f"case_id: {ex['case_id']}, category: {ex['category']}")
for msg in ex["messages"]:
    role = msg["role"]
    if msg.get("tool_calls"):
        for tc in msg["tool_calls"]:
            print(f"  [{role}] tool_call: {tc['function']['name']}(args_str={tc['function']['arguments']})")
    else:
        cont = msg["content"]
        text = " ".join(str(p.get("text") or p.get("path", ""))[:60] for p in cont if isinstance(p, dict))
        extra = f" (tool_call_id={msg.get('tool_call_id')}, name={msg.get('name')})" if role == "tool" else ""
        print(f"  [{role}] {text[:200]}{extra}")

SFT_DATA_ROOT = "/kaggle/working/sft_data"
os.makedirs(f"{SFT_DATA_ROOT}/train", exist_ok=True)
sft_rows = [{"messages": ex["messages"], "tools": ex["tools"]} for ex in train_examples]
ds = Dataset.from_list(sft_rows)
ds.to_parquet(f"{SFT_DATA_ROOT}/train/data.parquet")
print(f"\nWrote SFT parquet: {SFT_DATA_ROOT}/train/data.parquet ({len(sft_rows)} rows)")


In [ ]:
# DIAGNOSTIC: render the first trajectory via the model's chat_template (in subprocess
# because Kaggle's preloaded numpy 2.0.2 is stale; fresh subprocess gets numpy 2.2.6 from wheels)

# Pickle the messages + tools so the subprocess can read them
import pickle
DBG_DUMP = "/kaggle/working/dbg_render_input.pkl"
with open(DBG_DUMP, "wb") as f:
    pickle.dump({
        "messages": train_examples[0]["messages"],
        "tools":    json.loads(train_examples[0]["tools"]),
        "model_dir": MODEL_DIR,
    }, f)

DBG_SCRIPT = r'''
import pickle, sys
with open("/kaggle/working/dbg_render_input.pkl", "rb") as f:
    payload = pickle.load(f)
messages = payload["messages"]
tools    = payload["tools"]
MODEL_DIR = payload["model_dir"]

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

def strip_images(msgs):
    out = []
    for m in msgs:
        if isinstance(m.get("content"), list):
            new_content = [p for p in m["content"] if p.get("type") != "image"]
            out.append({**m, "content": new_content if new_content else [{"type": "text", "text": "[images omitted]"}]})
        else:
            out.append(m)
    return out

print("=== chat_template render WITHOUT tools arg ===")
try:
    rendered = tokenizer.apply_chat_template(strip_images(messages), tokenize=False, add_generation_prompt=False)
    print(f"length: {len(rendered)} chars")
    print(rendered[:3500])
    print()
    print(f"<|tool_call_start|> count: {rendered.count('<|tool_call_start|>')}")
    print(f"<|tool_call_end|>   count: {rendered.count('<|tool_call_end|>')}")
    print(f"<|im_start|>assistant count: {rendered.count('<|im_start|>assistant')}")
    print(f"<|im_start|>tool count: {rendered.count('<|im_start|>tool')}")
    print(f"compute_index_delta in render: {'compute_index_delta' in rendered}")
    print(f"submit_to_ground in render: {'submit_to_ground' in rendered}")
except Exception as e:
    import traceback; traceback.print_exc()
    print(f"FAIL: {type(e).__name__}: {e}")

print()
print("=== chat_template render WITH tools arg ===")
try:
    rendered2 = tokenizer.apply_chat_template(strip_images(messages), tools=tools, tokenize=False, add_generation_prompt=False)
    print(f"length: {len(rendered2)} chars")
    print(f"system tools_block ('List of tools' or 'tools' in first 500): "
          f"{'List of tools' in rendered2 or 'tools' in rendered2[:500].lower()}")
    print()
    print("--- first 1500 chars (for system prompt with tools) ---")
    print(rendered2[:1500])
except Exception as e:
    import traceback; traceback.print_exc()
    print(f"FAIL: {type(e).__name__}: {e}")
'''

with open("/kaggle/working/_dbg_render.py", "w") as f:
    f.write(DBG_SCRIPT)

import subprocess
result = subprocess.run(
    ["python3.12", "/kaggle/working/_dbg_render.py"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("=== STDERR (tail 2000) ===")
    print(result.stderr[-2000:])


In [ ]:
# DIAGNOSTIC c4c (FIXED): mimic prime-rl 0.4.0 build_loss_mask to see EXACTLY which tokens get loss

import pickle
DBG2_DUMP = "/kaggle/working/dbg_lossmask_input.pkl"
with open(DBG2_DUMP, "wb") as f:
    pickle.dump({
        "messages": train_examples[0]["messages"],
        "tools": json.loads(train_examples[0]["tools"]),
        "model_dir": MODEL_DIR,
    }, f)

DBG2_SCRIPT = r'''
import pickle, json, sys
with open("/kaggle/working/dbg_lossmask_input.pkl", "rb") as f:
    payload = pickle.load(f)
messages = payload["messages"]; tools = payload["tools"]; MODEL_DIR = payload["model_dir"]

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

def strip_images(msgs):
    out = []
    for m in msgs:
        if isinstance(m.get("content"), list):
            new_content = [p for p in m["content"] if p.get("type") != "image"]
            out.append({**m, "content": new_content if new_content else [{"type": "text", "text": ""}]})
        else:
            out.append(m)
    return out

def deserialize_tool_calls(msgs):
    def deserialize_tc(tc):
        return {**tc, "function": {**tc["function"],
                                   "arguments": json.loads(tc["function"]["arguments"])}}
    out = []
    for m in msgs:
        if m.get("tool_calls"):
            out.append({**m, "tool_calls": [deserialize_tc(tc) for tc in m["tool_calls"]]})
        else:
            out.append(m)
    return out

LOSS_MASK_CFG = {"system": False, "user": False, "assistant": True, "tool": False}
messages_safe = strip_images(deserialize_tool_calls(messages))

prev_ids = []
loss_mask = []
all_token_ids = []

for i, message in enumerate(messages_safe):
    if message["role"] == "tool" and i+1 < len(messages_safe) and messages_safe[i+1]["role"]=="tool":
        continue
    add_gen_prompt = (
        message["role"] in ["user", "tool"]
        and i + 1 < len(messages_safe)
        and messages_safe[i + 1]["role"] == "assistant"
    )
    # Use tokenize=False then manually encode (avoid Encoding obj issue)
    try:
        rendered = tokenizer.apply_chat_template(
            messages_safe[: i + 1], tools=tools,
            add_generation_prompt=add_gen_prompt, tokenize=False,
        )
        cur_ids = tokenizer.encode(rendered, add_special_tokens=False)
    except Exception as e:
        print(f"[turn {i}] FAIL: {type(e).__name__}: {e}")
        sys.exit(1)
    if prev_ids != cur_ids[:len(prev_ids)]:
        print(f"[turn {i}, role {message['role']}] PREFIX INVARIANT VIOLATED!")
        print(f"  prev_len={len(prev_ids)}, cur_len={len(cur_ids)}")
        for j in range(min(len(prev_ids), len(cur_ids))):
            if prev_ids[j] != cur_ids[j]:
                print(f"  diverge at {j}: prev={tokenizer.decode([prev_ids[j]])!r} cur={tokenizer.decode([cur_ids[j]])!r}")
                break
        sys.exit(1)
    new_tokens = cur_ids[len(prev_ids):]
    mask_value = LOSS_MASK_CFG.get(message["role"], False)
    new_text = tokenizer.decode(new_tokens, skip_special_tokens=False)
    print(f"\n[turn {i}, role={message['role']}, add_gen_prompt={add_gen_prompt}, n_new={len(new_tokens)}, masked={mask_value}]")
    print(f"  new text: {new_text[:400]!r}")
    loss_mask.extend([mask_value] * len(new_tokens))
    all_token_ids.extend(new_tokens)
    prev_ids = cur_ids

print(f"\n=== TOTALS ===")
print(f"total tokens: {len(all_token_ids)}")
print(f"masked True (loss applied): {sum(loss_mask)}")
print(f"masked False (no loss):     {len(loss_mask) - sum(loss_mask)}")

print(f"\n=== TOKENS WITH LOSS (decoded) ===")
loss_tokens = [tid for tid, m in zip(all_token_ids, loss_mask) if m]
print(f"n loss tokens: {len(loss_tokens)}")
loss_text = tokenizer.decode(loss_tokens, skip_special_tokens=False)
print(f"decoded: {loss_text!r}")

print(f"\n=== KEY VERIFICATION ===")
print(f"'compute_index_delta' in loss-mask text: {'compute_index_delta' in loss_text}")
print(f"'analyze' in loss-mask text:             {'analyze' in loss_text}")
print(f"'submit_to_ground' in loss-mask text:    {'submit_to_ground' in loss_text}")
'''

with open("/kaggle/working/_dbg_lossmask.py", "w") as f:
    f.write(DBG2_SCRIPT)

import subprocess
result = subprocess.run(["python3.12", "/kaggle/working/_dbg_lossmask.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("=== STDERR (tail 2000) ===")
    print(result.stderr[-2000:])


## 5. SFT TOML


In [ ]:
SFT_OUT  = "/kaggle/working/outputs_sft"
MAX_LEN  = 4096
SFT_STEPS = len(train_examples) * 3  # 3 epochs (was 1 epoch)
print(f"SFT steps planned: {SFT_STEPS} (= 3 epochs of {len(train_examples)} train_examples)")

SFT_TOML = f'''max_steps = {SFT_STEPS}
output_dir = "{SFT_OUT}"

[model]
name = "{MODEL_DIR}"
seq_len = {MAX_LEN}
attn = "sdpa"
optimization_dtype = "bfloat16"
reduce_dtype = "bfloat16"

[model.vlm]
vision_encoder_attr = "{VISION_ATTR}"
language_model_attr = "{LM_ATTR}"

[data]
name = "{SFT_DATA_ROOT}"
seq_len = {MAX_LEN}
batch_size = 1

[optim]
lr = 2e-5

[ckpt]
interval = {SFT_STEPS}

[ckpt.weights]
save_format = "safetensors"
save_sharded = false
'''

os.makedirs(SFT_OUT, exist_ok=True)
os.makedirs("/kaggle/working/outputs/proc_logs", exist_ok=True)
sft_toml_path = "/kaggle/working/outputs/sft.toml"
with open(sft_toml_path, "w") as f:
    f.write(SFT_TOML)
print("=== sft.toml ===")
print(SFT_TOML)


## 6. Run prime-rl SFT


In [ ]:
import time

env = os.environ.copy()
env.update({
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
    "WANDB_MODE": "disabled",
})

sft_log = "/kaggle/working/outputs/proc_logs/sft.log"
t0 = time.time()
with open(sft_log, "wb") as lf:
    result = subprocess.run(
        f"python3.12 -m prime_rl.entrypoints.sft @ {sft_toml_path}",
        shell=True, env=env, stdout=lf, stderr=subprocess.STDOUT,
    )
elapsed = time.time() - t0
print(f"\nSFT exit={result.returncode}  elapsed={elapsed:.1f}s")
print("\n--- sft.log tail ---")
subprocess.run(f"tail -n 80 {sft_log}", shell=True)
print("\n--- weights tree ---")
subprocess.run(f"find {SFT_OUT}/weights -maxdepth 3 -type f 2>/dev/null | head -30", shell=True)

if result.returncode != 0:
    raise RuntimeError(f"SFT failed exit={result.returncode}")
print("\nSFT PASS")


## 7. Multi-turn agent eval helper

vllm serve (custom parser plugin loaded) → 96 test cases で agent loop:
1. Send messages with tools
2. Parse tool_call from response
3. Execute tool (compute_index_delta = read precompute, analyze = ack, submit/drop = terminal)
4. Append assistant + tool messages, loop
5. max 6 turns safety, terminal on submit/drop


In [ ]:
import time, json, base64, signal, subprocess, urllib.request, urllib.error
import collections

EVAL_RESULTS = []


# Write sitecustomize.py so any Python subprocess auto-registers the LFM2 parser
SITECUSTOMIZE_DIR = "/kaggle/working/sitecust"
os.makedirs(SITECUSTOMIZE_DIR, exist_ok=True)
SITECUSTOMIZE_SRC = chr(10).join([
    "import sys",
    'sys.path.insert(0, "/kaggle/working/plugins")',
    "try:",
    "    from vllm.tool_parsers import ToolParserManager",
    '    ToolParserManager.import_tool_parser("/kaggle/working/plugins/lfm2_tool_parser.py")',
    '    print("[sitecustomize] registered lfm2_pythonic", flush=True)',
    "except Exception as _e:",
    '    print(f"[sitecustomize] WARN parser register: {type(_e).__name__}: {_e}", flush=True)',
    "",
    "# Monkey-patch satelliteagent_env to strip images from rollout prompts",
    "try:",
    "    import satelliteagent_env as _sae",
    "    _orig_load = _sae.load_environment",
    "    def _patched_load(*args, **kwargs):",
    "        env = _orig_load(*args, **kwargs)",
    "        if hasattr(env, 'dataset') and env.dataset is not None:",
    "            from datasets import Dataset",
    "            new_rows = []",
    "            for row in env.dataset:",
    "                msgs = row.get('prompt') or []",
    "                new_msgs = []",
    "                for m in msgs:",
    "                    cont = m.get('content')",
    "                    if isinstance(cont, list):",
    "                        new_cont = [p for p in cont if (p.get('type') if isinstance(p, dict) else None) not in ('image', 'image_url')]",
    "                        new_msgs.append({**m, 'content': new_cont if new_cont else 'No image. Use compute_index_delta to gather evidence.'})",
    "                    else:",
    "                        new_msgs.append(m)",
    "                new_rows.append({**row, 'prompt': new_msgs})",
    "            env.dataset = Dataset.from_list(new_rows)",
    '            print(f"[sitecustomize] patched satelliteagent_env: stripped images from {len(new_rows)} rollout rows", flush=True)',
    "        return env",
    "    _sae.load_environment = _patched_load",
    "except Exception as _e:",
    '    print(f"[sitecustomize] WARN env patch: {type(_e).__name__}: {_e}", flush=True)',
])
with open(f"{SITECUSTOMIZE_DIR}/sitecustomize.py", "w") as f:
    f.write(SITECUSTOMIZE_SRC)
print(f"Wrote {SITECUSTOMIZE_DIR}/sitecustomize.py")


def _b64_image(path):
    with open(path, "rb") as f:
        return "data:image/png;base64," + base64.b64encode(f.read()).decode("ascii")


def execute_tool(name, args, ex):
    case_id = ex["case_id"]
    if name == "compute_index_delta":
        feats = case_features(case_id)
        if feats is None: return "error: precompute missing"
        which = (args or {}).get("index", "all")
        if which == "all" or which is None: return build_observation(feats)
        if which in INDICES:
            d = feats[which]
            return f"{which}: mean={d.get('mean'):+.3f}, frac_decrease={d.get('frac_decrease_strong'):.3f}, frac_increase={d.get('frac_increase_strong'):.3f}"
        return f"error: unknown index {which!r}"
    if name == "analyze":
        return f"noted, call {(args or {}).get('recommended_action', '?')}"
    if name in ("submit_to_ground", "drop"): return "ok"
    return f"error: unknown tool {name!r}"


def _gpu_mem_mb():
    try:
        out = subprocess.check_output(
            "nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
            shell=True,
        ).decode().strip().splitlines()[0]
        return int(out)
    except Exception: return -1


def _wait_gpu_free(thr=8000, maxw=180):
    deadline = time.time() + maxw
    last = _gpu_mem_mb()
    while time.time() < deadline:
        u = _gpu_mem_mb()
        last = u
        if 0 <= u < thr: return u
        time.sleep(2)
    return last


def _hard_kill_vllm(p):
    try: os.killpg(os.getpgid(p.pid), signal.SIGTERM)
    except (ProcessLookupError, OSError): pass
    try: p.wait(timeout=5)
    except subprocess.TimeoutExpired: pass
    if p.poll() is None:
        try: os.killpg(os.getpgid(p.pid), signal.SIGKILL)
        except (ProcessLookupError, OSError): pass
        try: p.wait(timeout=5)
        except subprocess.TimeoutExpired: pass
    for pat in ["prime_rl.entrypoints.inference", "vllm.entrypoints", "vllm", "EngineCore", "MQLLMEngine"]:
        subprocess.run(f"pkill -9 -f '{pat}' 2>/dev/null || true", shell=True)


def start_vllm(model_path, label, port=8001):
    """Start vLLM and wait for ready. Returns (popen_process, log_path, served_id)."""
    print(f"\n=== starting vLLM [{label}] ===\nmodel_path = {model_path}")
    waited = _wait_gpu_free()
    print(f"GPU mem after wait = {waited} MB")
    INFER_TOML = f'''gpu_memory_utilization = 0.45

[model]
name = "{model_path}"
max_model_len = {MAX_LEN}
enforce_eager = true
tool_call_parser = "lfm2_pythonic"

[server]
port = {port}
'''
    toml_path = f"/kaggle/working/outputs/eval_{label}.toml"
    log_path  = f"/kaggle/working/outputs/proc_logs/eval_{label}.log"
    with open(toml_path, "w") as f: f.write(INFER_TOML)

    env_local = os.environ.copy()
    env_local.update({
        "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
        "WANDB_MODE": "disabled", "VLLM_API_KEY": "dummy",
        "CUDA_VISIBLE_DEVICES": "0",
        "PYTHONPATH": f"{SITECUSTOMIZE_DIR}:" + env_local.get("PYTHONPATH", ""),
    })
    lf = open(log_path, "wb")
    cmd = f"python3.12 -m prime_rl.entrypoints.inference @ {toml_path}"
    p = subprocess.Popen(cmd, shell=True, env=env_local, stdout=lf, stderr=subprocess.STDOUT, start_new_session=True)
    p._log_file = lf  # type: ignore[attr-defined]
    p._log_path = log_path  # type: ignore[attr-defined]

    ready, deadline = False, time.time() + 600
    while time.time() < deadline:
        if p.poll() is not None:
            _hard_kill_vllm(p)
            raise RuntimeError(f"vLLM died early. Tail:\n{subprocess.check_output(f'tail -n 60 {log_path}', shell=True).decode(errors='replace')}")
        try:
            req = urllib.request.Request(f"http://localhost:{port}/v1/models",
                headers={"Authorization": "Bearer dummy"})
            if urllib.request.urlopen(req, timeout=5).status == 200:
                ready = True; break
        except Exception: pass
        time.sleep(5)
    if not ready:
        _hard_kill_vllm(p)
        raise RuntimeError(f"vLLM not ready: {subprocess.check_output(f'tail -n 80 {log_path}', shell=True).decode(errors='replace')}")
    print(f"[{label}] vLLM up, GPU mem = {_gpu_mem_mb()} MB")
    try:
        with urllib.request.urlopen(urllib.request.Request(
                f"http://localhost:{port}/v1/models",
                headers={"Authorization": "Bearer dummy"}), timeout=10) as r:
            served_id = json.loads(r.read())["data"][0]["id"]
    except Exception:
        served_id = model_path
    print(f"[{label}] served id = {served_id}")
    return p, served_id


def stop_vllm(p, label):
    _hard_kill_vllm(p)
    try: p._log_file.close()
    except Exception: pass
    freed = _wait_gpu_free(thr=8000, maxw=180)
    print(f"[{label}] GPU mem after vLLM stop = {freed} MB")


def chat_complete(port, served_id, messages, *, max_tokens=256, temperature=0.0,
                  tool_choice="required"):
    """Single /v1/chat/completions call. Returns full parsed response dict.

    Default tool_choice='required' matches S10/S40-S45 (GRPO eval) — this forces
    vLLM grammar-constrained generation to emit tool_calls in lfm2_pythonic format.
    """
    body = {
        "model": served_id, "messages": messages,
        "tools": TOOLS, "tool_choice": tool_choice,
        "max_tokens": max_tokens, "temperature": temperature,
    }
    req = urllib.request.Request(
        f"http://localhost:{port}/v1/chat/completions",
        data=json.dumps(body).encode("utf-8"),
        headers={"Content-Type": "application/json", "Authorization": "Bearer dummy"},
    )
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.loads(r.read())


def initial_messages(case_id):
    before = f"{DATA_ROOT}/curated_pairs/{case_id}/before.png"
    after  = f"{DATA_ROOT}/curated_pairs/{case_id}/after.png"
    return [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user", "content": [
            {"type": "text",       "text": "Before image (previous pass):"},
            {"type": "image_url",  "image_url": {"url": _b64_image(before)}},
            {"type": "text",       "text": "After image (current pass):"},
            {"type": "image_url",  "image_url": {"url": _b64_image(after)}},
            {"type": "text",       "text": "Investigate with your tools, then decide submit_to_ground or drop."},
        ]},
    ]


def initial_messages_no_image(case_id):
    """Like initial_messages but strips out the {image_url} entries — keeps only text."""
    msgs = initial_messages(case_id)
    new = []
    for m in msgs:
        if isinstance(m.get("content"), list):
            new_content = [p for p in m["content"] if p.get("type") != "image_url"]
            new.append({**m, "content": new_content})
        else:
            new.append(m)
    return new


def smoke_test_one(port, served_id, label, n_to_print=2, use_images=True):
    """Send one request per category (up to N), print full response details."""
    print(f"\n--- SMOKE TEST [{label}] ---")
    seen_cats = set()
    for ex in test_examples:
        if ex["category"] in seen_cats: continue
        seen_cats.add(ex["category"])
        if len(seen_cats) > n_to_print: break

        cid = ex["case_id"]
        msgs = initial_messages(cid) if use_images else initial_messages_no_image(cid)
        try:
            resp = chat_complete(port, served_id, msgs, max_tokens=256)
        except Exception as e:
            print(f"  [{cid}] HTTP FAIL: {type(e).__name__}: {e}")
            continue
        choice = resp["choices"][0]
        msg = choice["message"]
        finish = choice.get("finish_reason")
        content = msg.get("content")
        tcs = msg.get("tool_calls") or []
        print(f"\n  case={cid} category={ex['category']}")
        print(f"    finish_reason={finish}")
        print(f"    content={content!r}")
        print(f"    n_tool_calls={len(tcs)}")
        for tc in tcs:
            fn = tc.get("function", {})
            print(f"    tool_call: {fn.get('name')}({fn.get('arguments')})")
        if not tcs and content:
            print(f"    (raw text mode — parser saw NO tool_call)")


def run_agent_eval(model_path, label, *, port=8001, max_turns=6, n_cases=None,
                   smoke_only=False, save_responses=True, use_images=True):
    p, served_id = start_vllm(model_path, label, port=port)
    try:
        smoke_test_one(port, served_id, label, n_to_print=3, use_images=use_images)
        if smoke_only:
            return None

        cases = test_examples[:n_cases] if n_cases else test_examples
        per_cat = collections.defaultdict(lambda: [0, 0])
        confusion = collections.Counter()
        per_case = []
        all_responses = []
        t0 = time.time()
        for i, ex in enumerate(cases):
            cid = ex["case_id"]
            messages = initial_messages(cid) if use_images else initial_messages_no_image(cid)
            terminal, tool_call_log, raw_log = None, [], []
            for turn in range(max_turns):
                try:
                    resp = chat_complete(port, served_id, messages, max_tokens=256)
                except Exception as e:
                    terminal = f"HTTP_ERR_{type(e).__name__}"
                    raw_log.append({"turn": turn, "error": str(e)})
                    break
                msg = resp["choices"][0]["message"]
                tcs = msg.get("tool_calls") or []
                content = msg.get("content")
                finish = resp["choices"][0].get("finish_reason")
                raw_log.append({
                    "turn": turn,
                    "finish": finish,
                    "content_preview": (content or "")[:200] if isinstance(content, str) else content,
                    "n_tool_calls": len(tcs),
                    "tool_calls": [{"name": tc.get("function", {}).get("name"),
                                    "args_str": tc.get("function", {}).get("arguments", "")[:200]}
                                   for tc in tcs],
                })
                if not tcs:
                    terminal = "drop"
                    break
                messages.append({"role": "assistant", "content": content or "", "tool_calls": tcs})
                tc = tcs[0]
                fn = tc.get("function", {})
                fname = fn.get("name")
                try: fargs = json.loads(fn.get("arguments") or "{}")
                except Exception: fargs = {}
                tool_call_log.append({"name": fname, "args": fargs})
                if fname in ("submit_to_ground", "drop"):
                    terminal = fname
                    break
                obs = execute_tool(fname, fargs, ex)
                messages.append({"role": "tool", "tool_call_id": tc.get("id", f"t{turn}"),
                                 "name": fname, "content": str(obs)})
            if terminal is None: terminal = "max_turns_exceeded"

            expected = "submit_to_ground" if ex["category"].startswith("pos_") else "drop"
            ok = (terminal == expected)
            per_cat[ex["category"]][1] += 1
            if ok: per_cat[ex["category"]][0] += 1
            confusion[(expected, terminal)] += 1
            entry = {
                "case_id": cid, "category": ex["category"],
                "expected": expected, "got": terminal, "ok": ok,
                "n_turns": len(tool_call_log), "tool_calls": tool_call_log,
                "raw_log": raw_log,
            }
            per_case.append(entry)
            if save_responses: all_responses.append(entry)
            if (i + 1) % 10 == 0:
                acc = sum(1 for c in per_case if c["ok"]) / len(per_case)
                print(f"  [{i+1}/{len(cases)}] acc={acc:.1%} elapsed {time.time()-t0:.0f}s")

        elapsed = time.time() - t0
        n = len(per_case); correct = sum(1 for c in per_case if c["ok"])
        summary = {
            "label": label, "model_path": model_path,
            "n": n, "correct": correct, "acc": correct/n if n else 0.0,
            "by_cat": {cat: {"n": v[1], "correct": v[0], "acc": v[0]/v[1] if v[1] else 0}
                       for cat, v in per_cat.items()},
            "confusion": {f"{k[0]}->{k[1]}": v for k, v in confusion.items()},
            "elapsed_s": round(elapsed, 1),
        }
        EVAL_RESULTS.append(summary)
        if save_responses:
            outp = f"/kaggle/working/outputs/eval_{label}_responses.jsonl"
            with open(outp, "w") as f:
                for e in all_responses:
                    f.write(json.dumps(e, default=str) + "\n")
            print(f"saved {len(all_responses)} responses -> {outp}")

        print(f"\n=== {label} overall: {correct}/{n} = {summary['acc']:.1%}  (elapsed {elapsed:.0f}s) ===")
        print(f"{'category':<22} {'n':>4} {'correct':>8} {'acc%':>8}")
        for cat, d in sorted(summary["by_cat"].items()):
            print(f"{cat:<22} {d['n']:>4} {d['correct']:>8} {d['acc']*100:>7.1f}%")
        print(f"\n{label} confusion:")
        for k, v in sorted(summary["confusion"].items(), key=lambda x: -x[1]):
            print(f"  {k}  {v}")

        # Print first 3 detailed traces
        print(f"\n=== first 3 case traces [{label}] ===")
        for entry in per_case[:3]:
            print(f"\n  case={entry['case_id']} cat={entry['category']} expected={entry['expected']} got={entry['got']} ok={entry['ok']}")
            for r in entry["raw_log"]:
                print(f"    turn{r['turn']}: finish={r.get('finish')} n_tcs={r['n_tool_calls']}")
                if r.get("content_preview"):
                    print(f"      content: {r['content_preview']!r}")
                for tc in r.get("tool_calls", []):
                    print(f"      tc: {tc['name']}({tc['args_str']})")

        return summary
    finally:
        stop_vllm(p, label)


## 8. Eval BASE (no SFT)


In [ ]:
run_agent_eval(MODEL_DIR, label="base")


## 8b. Eval BASE (no images)


In [ ]:
run_agent_eval(MODEL_DIR, label="base_no_image", use_images=False)


## 9. Eval AFTER SFT


In [ ]:
import shutil
weights_root = f"{SFT_OUT}/weights"
step_dirs = sorted(
    [d for d in os.listdir(weights_root) if d.startswith("step_")],
    key=lambda d: int(d.split("_")[-1]),
)
SFT_CKPT_DIR = f"{weights_root}/{step_dirs[-1]}"
print(f"SFT ckpt: {SFT_CKPT_DIR}")

# Copy processor / tokenizer files from base model into SFT ckpt dir
WEIGHT_EXTS = (".safetensors", ".bin", ".pth", ".pt", ".index.json")
SFT_OWNS    = {"config.json"}
copied = []
for fname in os.listdir(MODEL_DIR):
    src = f"{MODEL_DIR}/{fname}"
    dst = f"{SFT_CKPT_DIR}/{fname}"
    if not os.path.isfile(src): continue
    if any(fname.endswith(ext) for ext in WEIGHT_EXTS): continue
    if fname in SFT_OWNS: continue
    if os.path.exists(dst): continue
    shutil.copy(src, dst)
    copied.append(fname)
print(f"copied processor files: {copied}")

run_agent_eval(SFT_CKPT_DIR, label="after_sft")


## 9b. Eval AFTER SFT (no images) — IMAGE ATTENTION TEST


In [ ]:
run_agent_eval(SFT_CKPT_DIR, label="after_sft_no_image", use_images=False)


## 9c. GRPO setup (3-process, no-image rollouts via env monkey-patch)

SFT checkpoint init で GRPO 30 step。 sitecustomize が satelliteagent_env を patch するので
rollout は no-image 設定で動く (orchestrator も sitecustomize 経由で patch 適用)。


In [ ]:
GRPO_BASE_MODEL = SFT_CKPT_DIR  # init from no-image-trained SFT
GRPO_STEPS = 30
GRPO_OUT = "/kaggle/working/outputs"

TRAINER_TOML = f'''max_steps = {GRPO_STEPS}

[model]
name = "{GRPO_BASE_MODEL}"
seq_len = {MAX_LEN}
attn = "sdpa"
optimization_dtype = "bfloat16"
reduce_dtype = "bfloat16"

[model.vlm]
vision_encoder_attr = "{VISION_ATTR}"
language_model_attr = "{LM_ATTR}"

[optim]
lr = 1e-6

[ckpt]
interval = {GRPO_STEPS}

[ckpt.weights]
save_format = "safetensors"
save_sharded = false
'''

INFER_TOML = f'''gpu_memory_utilization = 0.5

[model]
name = "{GRPO_BASE_MODEL}"
max_model_len = {MAX_LEN}
enforce_eager = true
tool_call_parser = "lfm2_pythonic"

[server]
port = 8000
'''

ORCH_TOML = f'''max_steps = {GRPO_STEPS}
batch_size = 4
seq_len = {MAX_LEN}
rollouts_per_example = 2
filters = []

[model]
name = "{GRPO_BASE_MODEL}"

[train.sampling]
max_completion_tokens = 256

[train.sampling.extra_body]
tool_choice = "required"

[[train.env]]
id = "satelliteagent_env"

[train.env.args]
toy = false
data_root = "{DATA_ROOT}"
'''

trainer_toml = f"{GRPO_OUT}/trainer.toml"
infer_toml   = f"{GRPO_OUT}/infer.toml"
orch_toml    = f"{GRPO_OUT}/orch.toml"
with open(trainer_toml, "w") as f: f.write(TRAINER_TOML)
with open(infer_toml,   "w") as f: f.write(INFER_TOML)
with open(orch_toml,    "w") as f: f.write(ORCH_TOML)

for name, body in [("trainer.toml", TRAINER_TOML), ("infer.toml", INFER_TOML), ("orch.toml", ORCH_TOML)]:
    print(f"=== {name} ===")
    print(body)


In [ ]:
import os, subprocess, time, urllib.request

LOG_DIR = "/kaggle/working/outputs/proc_logs"
os.makedirs(LOG_DIR, exist_ok=True)
API_KEY = "dummy"

env_grpo = os.environ.copy()
env_grpo.update({
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
    "WANDB_MODE": "disabled",
    "CUDA_VISIBLE_DEVICES": "0",
    "VLLM_API_KEY": API_KEY,
    "PYTHONPATH": f"{SITECUSTOMIZE_DIR}:" + env_grpo.get("PYTHONPATH", ""),
})

procs = {}
def start_bg(name, cmd):
    log_path = f"{LOG_DIR}/{name}.log"
    log_f = open(log_path, "wb")
    p = subprocess.Popen(cmd, shell=True, env=env_grpo, stdout=log_f, stderr=subprocess.STDOUT,
                         start_new_session=True)
    procs[name] = (p, log_f, log_path)
    print(f"[{name}] started pid={p.pid} log={log_path}")
    return p

def cleanup_grpo():
    for name, (p, f, _) in procs.items():
        if p.poll() is None:
            try:
                os.killpg(os.getpgid(p.pid), 15)
                p.wait(timeout=10)
            except Exception:
                try: p.kill()
                except Exception: pass
        f.close()
    for pat in ["prime_rl.entrypoints.inference", "prime_rl.orchestrator",
                "prime_rl.trainer.rl", "vllm.entrypoints", "vllm",
                "EngineCore", "MQLLMEngine"]:
        subprocess.run(f"pkill -9 -f '{pat}' 2>/dev/null || true", shell=True)

def tail(path, n=80):
    try: return subprocess.check_output(f"tail -n {n} {path}", shell=True).decode(errors="replace")
    except Exception as e: return f"<tail error: {e}>"

outcome = "FAIL"
try:
    start_bg("inference", f"python3.12 -m prime_rl.entrypoints.inference @ {infer_toml}")
    print("Waiting for vLLM inference (max 10 min) ...")
    ready = False; deadline = time.time() + 600
    while time.time() < deadline:
        if procs["inference"][0].poll() is not None:
            raise RuntimeError(f"inference died early. Tail:\n{tail(procs['inference'][2], 80)}")
        try:
            req = urllib.request.Request(
                "http://localhost:8000/v1/models",
                headers={"Authorization": f"Bearer {API_KEY}"},
            )
            if urllib.request.urlopen(req, timeout=5).status == 200:
                ready = True; break
        except Exception: pass
        time.sleep(5)
    if not ready:
        raise RuntimeError(f"inference not ready in 10min. Tail:\n{tail(procs['inference'][2], 80)}")
    print("inference is up.")

    start_bg("orchestrator", f"python3.12 -m prime_rl.orchestrator.orchestrator @ {orch_toml}")
    time.sleep(5)
    if procs["orchestrator"][0].poll() is not None:
        raise RuntimeError(f"orchestrator died early. Tail:\n{tail(procs['orchestrator'][2], 80)}")
    print("orchestrator is up.")

    print("\nStarting trainer (live via tee)...")
    print("=" * 60)
    trainer_log = f"{LOG_DIR}/trainer.log"
    t0 = time.time()
    trainer_cmd = (
        "set -o pipefail; "
        "python3.12 -m torch.distributed.run --nproc-per-node 1 "
        f"-m prime_rl.trainer.rl.train @ {trainer_toml} "
        f"2>&1 | tee {trainer_log}"
    )
    trainer = subprocess.run(["bash", "-c", trainer_cmd], env=env_grpo)
    elapsed = time.time() - t0
    print("=" * 60)
    print(f"\ntrainer exit={trainer.returncode}  elapsed={elapsed:.1f}s")
    if trainer.returncode != 0:
        print("\n--- inference tail ---");    print(tail(procs["inference"][2], 60))
        print("\n--- orchestrator tail ---"); print(tail(procs["orchestrator"][2], 60))
        raise RuntimeError(f"trainer failed exit={trainer.returncode}")
    outcome = "PASS"
    print("\nGRPO PASS")
finally:
    cleanup_grpo()
    print(f"GRPO outcome: {outcome}")


In [ ]:
import shutil
GRPO_WEIGHTS_ROOT = f"{GRPO_OUT}/weights"
grpo_steps = sorted(
    [d for d in os.listdir(GRPO_WEIGHTS_ROOT) if d.startswith("step_")],
    key=lambda d: int(d.split("_")[-1]),
) if os.path.isdir(GRPO_WEIGHTS_ROOT) else []

if not grpo_steps:
    print("WARN: no GRPO weights, skipping after_grpo eval")
else:
    GRPO_CKPT_DIR = f"{GRPO_WEIGHTS_ROOT}/{grpo_steps[-1]}"
    print(f"GRPO ckpt: {GRPO_CKPT_DIR}")
    WEIGHT_EXTS = (".safetensors", ".bin", ".pth", ".pt", ".index.json")
    SFT_OWNS    = {"config.json"}
    for fname in os.listdir(MODEL_DIR):
        src = f"{MODEL_DIR}/{fname}"
        dst = f"{GRPO_CKPT_DIR}/{fname}"
        if not os.path.isfile(src): continue
        if any(fname.endswith(ext) for ext in WEIGHT_EXTS): continue
        if fname in SFT_OWNS: continue
        if os.path.exists(dst): continue
        shutil.copy(src, dst)

    run_agent_eval(GRPO_CKPT_DIR, label="after_grpo_no_image", use_images=False)


## 10. Summary + cleanup


In [ ]:
print("\n" + "=" * 60)
print("SUMMARY (multi-turn agent eval, n=" + str(len(test_examples)) + ")")
print("=" * 60)
print(f"{'stage':<14} {'overall':>10}")
for r in EVAL_RESULTS:
    print(f"{r['label']:<14} {r['acc']:>9.1%} ({r['correct']}/{r['n']})")
print()
print(f"{'category':<22}", end="")
for r in EVAL_RESULTS:
    print(f" {r['label'][:10]:>10}", end="")
print()
all_cats = sorted(set(c for r in EVAL_RESULTS for c in r["by_cat"]))
for cat in all_cats:
    print(f"{cat:<22}", end="")
    for r in EVAL_RESULTS:
        d = r["by_cat"].get(cat, {})
        print(f" {d.get('acc', 0)*100:>9.1f}%", end="")
    print()

with open("/kaggle/working/outputs/eval_results.json", "w") as f:
    json.dump(EVAL_RESULTS, f, indent=2, ensure_ascii=False, default=str)
print("\nsaved: /kaggle/working/outputs/eval_results.json")

# Cleanup heavy outputs (keep eval JSON only)
for victim in [
    f"{SFT_OUT}/weights",
    f"{SFT_OUT}/checkpoints",
    "/kaggle/working/sft_data",
]:
    if os.path.isdir(victim):
        shutil.rmtree(victim)
        print(f"removed: {victim}")
